# Capital Nerve — End-to-End Pipeline

One cell per service layer. Run top-to-bottom; each step consumes the previous event and emits the next.

```text
┌────────────────────────────┐
│ Source Discovery Layer      │
│ NSE/BSE/IR sites/manual     │
└──────────────┬─────────────┘
               │ filing.detected
               ▼
┌────────────────────────────┐
│ Document Ingestion Service  │
│ download, hash, store       │
└──────────────┬─────────────┘
               │ document.stored
               ▼
┌────────────────────────────┐
│ Parsing / Markdown Service  │
│ PDF → markdown/chunks       │
└──────────────┬─────────────┘
               │ document.parsed
               ▼
┌────────────────────────────┐
│ Extraction Service          │
│ SLM/LLM/OCR/table parser    │
└──────────────┬─────────────┘
               │ values.extracted
               ▼
┌────────────────────────────┐
│ Validation / Trust Service  │
│ evidence, formula, checks   │
└──────────────┬─────────────┘
               │ values.validated
               ▼
┌────────────────────────────┐
│ Fact Store (SQLite)       │
│ upsert quarters, history    │
└──────────────┬─────────────┘
               │
               ▼
┌────────────────────────────┐
│ Metrics Engine              │
│ catalog/metrics.json        │
│ filing first, DB fallback   │
└──────────────┬─────────────┘
               │ metrics.calculated
               ▼
┌────────────────────────────┐
│ Signal Engine               │
│ interpretation layer        │
└──────────────┬─────────────┘
               │ signals.generated
               ▼
┌────────────────────────────┐
│ Card Engine                 │
│ structured intelligence     │
└──────────────┬─────────────┘
               │ cards.generated
               ▼
┌────────────────────────────┐
│ Serving Layer               │
│ UI / API / alerts / search  │
└────────────────────────────┘
```

**Rule:** SLM reads language. Code does math. Database stores truth.

Validated facts persist to `data/capital_nerve.db`. Re-uploading a quarter upserts; prior quarters remain for QoQ/trends.

In [1]:
# ## Delete fact_values and filing where source_document_id/document_id = 'doc_8387fd727998'
# import sqlite3

# db_path = "data/capital_nerve.db"
# source_document_id = "doc_0d6cc67c4e68"

# with sqlite3.connect(db_path) as conn:
#     cur = conn.cursor()
#     # Delete from fact_values
#     cur.execute(
#         """
#         DELETE FROM fact_values
#         WHERE source_document_id = ?
#         """,
#         (source_document_id,)
#     )
#     # Delete from filing
#     cur.execute(
#         """
#         DELETE FROM filings
#         WHERE document_id = ?
#         """,
#         (source_document_id,)
#     )
#     conn.commit()
# print(f"Rows deleted successfully from fact_values and filing for document_id/source_document_id = {source_document_id}.")

In [2]:
# def delete_company_by_ticker(company_ticker: str, db_path: str = "data/capital_nerve.db") -> None:
#     """
#     Delete all filings and related fact_values for a company, identified via its company_ticker.
#     Looks up all document_ids for the given company_ticker in filings, then deletes all related rows.
#     Args:
#         company_ticker: The company ticker (e.g., 'TCS').
#         db_path: Path to the SQLite database.
#     """
#     import sqlite3

#     with sqlite3.connect(db_path) as conn:
#         cur = conn.cursor()

#         # Find all document_ids for the company_ticker from filings
#         cur.execute(
#             "SELECT document_id FROM filings WHERE company_ticker = ?",
#             (company_ticker,)
#         )
#         doc_ids = [row[0] for row in cur.fetchall()]

#         if not doc_ids:
#             print(f"No filings found for company_ticker '{company_ticker}'. Nothing to delete.")
#             return

#         # Delete all fact_values for those document_ids as source_document_id
#         cur.executemany(
#             "DELETE FROM fact_values WHERE source_document_id = ?",
#             [(doc_id,) for doc_id in doc_ids]
#         )

#         # Delete all filings for those document_ids
#         cur.executemany(
#             "DELETE FROM filings WHERE document_id = ?",
#             [(doc_id,) for doc_id in doc_ids]
#         )

#         conn.commit()
#     print(f"Deleted all filings and fact_values for company_ticker '{company_ticker}'.")

# # Example usage:
# delete_company_by_ticker("AXISBANK")

In [3]:
# def delete_company_by_ticker(company_ticker: str, db_path: str = "data/capital_nerve.db") -> None:
#     """
#     Delete all filings and related metric_values for a company, identified via its company_ticker.
#     Looks up all document_ids for the given company_ticker in filings, then deletes all related rows.
#     Args:
#         company_ticker: The company ticker (e.g., 'TCS').
#         db_path: Path to the SQLite database.
#     """
#     import sqlite3

#     with sqlite3.connect(db_path) as conn:
#         cur = conn.cursor()

#         # Use parameterized query to safely bind company_ticker as a value, not a column
#         cur.execute(
#             """
#             DELETE FROM fact_values WHERE company_ticker = ?
#             """,
#             (company_ticker,)
#         )

#         conn.commit()

#     print(f"Deleted all fact_values for company_ticker '{company_ticker}'.")

# # Example usage:
# delete_company_by_ticker("AXISBANK")

In [4]:
# Document ingestion from the uploads folder. 
# All documents in the data/uploads directory are available for processing.

from pathlib import Path

DATA_DIR = Path("data")
UPLOAD_DIR = DATA_DIR / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ingesting from uploads folder: {UPLOAD_DIR.resolve()}")
files = list(UPLOAD_DIR.glob("*"))
if not files:
    print("No files found in the uploads folder. Please add files to this folder to proceed.")
else:
    print(f"Found {len(files)} file(s) in uploads folder:")
    for f in files:
        print(f" - {f.name} ({f.stat().st_size:,} bytes)")

Ingesting from uploads folder: /Users/prairit/capital-nerve/v2/data/uploads
Found 8 file(s) in uploads folder:
 - Q1-FY2024-25.pdf (1,253,337 bytes)
 - Q2-FY2024-25.pdf (1,664,783 bytes)
 - Q4_FY2025-26.pdf (986,633 bytes)
 - Q4_FY2024_25.pdf (1,719,021 bytes)
 - Q3_FY2025-26.pdf (766,847 bytes)
 - Q2_FY2025-26.pdf (995,466 bytes)
 - Q1_FY2025-26.pdf (719,459 bytes)
 - Q3_FY2024-25.pdf (1,855,556 bytes)


In [5]:
# Bootstrap — paths, events, FilingCandidate (run once per session)

from __future__ import annotations

import hashlib
import json
import re
import uuid
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
load_dotenv(NOTEBOOK_DIR / ".env")

DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR = DATA_DIR / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


def _truncate_for_display(obj: Any, max_str: int = 240, max_list: int = 8) -> Any:
    if isinstance(obj, str):
        if len(obj) <= max_str:
            return obj
        return f"{obj[:max_str]}… [{len(obj)} chars total]"
    if isinstance(obj, dict):
        return {k: _truncate_for_display(v, max_str, max_list) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        items = list(obj)
        shown = [_truncate_for_display(x, max_str, max_list) for x in items[:max_list]]
        if len(items) > max_list:
            shown.append(f"… +{len(items) - max_list} more item(s)")
        return shown
    return obj


def print_cell_output(step: str, data: Any) -> None:
    """Pretty-print cell results for verification (long text/lists truncated)."""
    print(f"\n── {step} ──")
    print(json.dumps(_truncate_for_display(data), indent=2, default=str))


def emit(event_type: str, payload: dict[str, Any]) -> dict[str, Any]:
    event = {
        "event_type": event_type,
        "event_id": new_id("evt"),
        "occurred_at": now_iso(),
        "payload": payload,
    }
    print(json.dumps(
        {
            "event_type": event_type,
            "event_id": event["event_id"],
            "occurred_at": event["occurred_at"],
        },
        indent=2,
    ))
    print_cell_output(f"{event_type} — cell output", payload)
    return event


@dataclass
class FilingCandidate:
    source: str  # nse | bse | ir | manual
    company_ticker: str
    title: str
    url: str | None = None
    local_path: str | None = None
    detected_at: str = field(default_factory=now_iso)


def discover_filings() -> list[FilingCandidate]:
    """Discover uploaded/manual filings from the uploads directory for ingestion."""

    # Prefer uploaded files if available, otherwise provide a demo/sample
    files = list(UPLOAD_DIR.glob("*"))
    candidates = []
    if files:
        for f in files:
            candidates.append(
                FilingCandidate(
                    source="manual",
                    company_ticker="Reliance Industries Ltd.",
                    title=f.name,
                    local_path=str(f.resolve()),
                )
            )
        return candidates
    else:
        # No files found in uploads, return a text message
        return "No document found in upload folder"


from periods import (
    ReportingPeriod,
    detect_reporting_period,
    prior_period_match_score,
    prior_quarter_match_score,
    prior_quarter_period,
    prior_year_period,
    period_match_score,
    reporting_period_from_date,
    resolve_period_label,
)
PREFERRED_METRIC_BASIS = "consolidated"


def detect_available_bases(markdown: str) -> list[str]:
    found: list[str] = []
    if re.search(r"\bstandalone\b", markdown, re.I):
        found.append("standalone")
    if re.search(r"\bconsolidated\b", markdown, re.I):
        found.append("consolidated")
    return found


def preferred_metric_basis(markdown: str) -> str:
    available = detect_available_bases(markdown)
    if "consolidated" in available:
        return "consolidated"
    if "standalone" in available:
        return "standalone"
    return PREFERRED_METRIC_BASIS


def basis_from_heading(heading: str | None) -> str | None:
    if not heading:
        return None
    h = heading.lower()
    if "consolidated" in h:
        return "consolidated"
    if "standalone" in h:
        return "standalone"
    return None


def basis_match_score(value_basis: str | None, preferred: str) -> int:
    if not value_basis:
        return 50
    b = value_basis.strip().lower()
    if b == preferred:
        return 100
    if b in ("standalone", "consolidated"):
        return -1
    return 0


def _row_selection_score(
    row: dict[str, Any],
    target: ReportingPeriod,
    preferred_basis: str,
    *,
    prior_year: bool = False,
    prior_quarter: bool = False,
) -> int:
    if row.get("status") != "accepted":
        return -1
    if prior_year:
        period_score = prior_period_match_score(row.get("period"), target)
    elif prior_quarter:
        period_score = prior_quarter_match_score(row.get("period"), target)
    else:
        period_score = period_match_score(row.get("period"), target)
    if period_score <= 0:
        return -1
    if basis_match_score(row.get("basis"), preferred_basis) < 0:
        return -1
    return period_score + basis_match_score(row.get("basis"), preferred_basis)


def _select_metrics(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str,
    *,
    prior_year: bool = False,
    prior_quarter: bool = False,
    allow_fallback: bool = True,
) -> dict[str, float]:
    best: dict[str, tuple[int, float]] = {}
    for row in values:
        score = _row_selection_score(
            row,
            target,
            preferred_basis,
            prior_year=prior_year,
            prior_quarter=prior_quarter,
        )
        if score <= 0:
            continue
        key = row["fact_key"]
        val = float(row["numeric_value"])
        prev = best.get(key)
        if prev is None or score > prev[0]:
            best[key] = (score, val)
    result = {k: v for k, (_, v) in best.items()}
    if result or not allow_fallback or preferred_basis != "consolidated":
        return result
    return _select_metrics(
        values,
        target,
        "standalone",
        prior_year=prior_year,
        prior_quarter=prior_quarter,
        allow_fallback=False,
    )


def select_metrics_for_period(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str = PREFERRED_METRIC_BASIS,
) -> dict[str, float]:
    return _select_metrics(values, target, preferred_basis, prior_year=False)


def select_prior_year_metrics(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str = PREFERRED_METRIC_BASIS,
) -> dict[str, float]:
    return _select_metrics(values, target, preferred_basis, prior_year=True)


def select_prior_quarter_metrics(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str = PREFERRED_METRIC_BASIS,
) -> dict[str, float]:
    return _select_metrics(values, target, preferred_basis, prior_quarter=True)


def find_best_value_row(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str,
    fact_key: str,
    *,
    prior_year: bool = False,
    prior_quarter: bool = False,
) -> dict[str, Any] | None:
    best_row: dict[str, Any] | None = None
    best_score = -1
    for basis in (preferred_basis, "standalone") if preferred_basis == "consolidated" else (preferred_basis,):
        for row in values:
            if row.get("fact_key") != fact_key:
                continue
            score = _row_selection_score(
                row,
                target,
                basis,
                prior_year=prior_year,
                prior_quarter=prior_quarter,
            )
            if score > best_score:
                best_score = score
                best_row = row
        if best_row is not None:
            break
    return best_row if best_score > 0 else None


def resolve_input_provenance(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str,
    store: Any,
    company_ticker: str,
    fact_key: str,
    *,
    prior_year: bool = False,
    prior_quarter: bool = False,
) -> dict[str, Any] | None:
    if prior_year:
        lookup = prior_year_period(target)
    elif prior_quarter:
        lookup = prior_quarter_period(target)
    else:
        lookup = target

    row = find_best_value_row(
        values,
        target,
        preferred_basis,
        fact_key,
        prior_year=prior_year,
        prior_quarter=prior_quarter,
    )
    if row:
        return {
            "fact_key": fact_key,
            "value": float(row["numeric_value"]),
            "period": lookup.label,
            "evidence": row.get("evidence"),
            "source": "filing",
            "value_id": row.get("value_id"),
            "unit": row.get("unit"),
        }

    detail = store.load_fact_detail(
        company_ticker,
        lookup.quarter,
        lookup.fy_start_year,
        preferred_basis,
        fact_key,
    )
    if not detail and preferred_basis == "consolidated":
        detail = store.load_fact_detail(
            company_ticker,
            lookup.quarter,
            lookup.fy_start_year,
            "standalone",
            fact_key,
        )
    if not detail:
        return None
    return {
        "fact_key": fact_key,
        "value": detail["numeric_value"],
        "period": lookup.label,
        "evidence": detail.get("evidence"),
        "source": "database",
        "value_id": None,
        "unit": detail.get("unit"),
    }


def resolve_metrics(
    values: list[dict[str, Any]],
    target: ReportingPeriod,
    preferred_basis: str,
    store: Any,
    company_ticker: str,
    *,
    prior_year: bool = False,
    prior_quarter: bool = False,
) -> dict[str, float]:
    if prior_year:
        local = select_prior_year_metrics(values, target, preferred_basis)
        lookup = prior_year_period(target)
    elif prior_quarter:
        local = select_prior_quarter_metrics(values, target, preferred_basis)
        lookup = prior_quarter_period(target)
    else:
        local = select_metrics_for_period(values, target, preferred_basis)
        lookup = target
    if local:
        return local
    db_facts = store.load_facts(
        company_ticker, lookup.quarter, lookup.fy_start_year, preferred_basis
    )
    if db_facts:
        return db_facts
    if preferred_basis == "consolidated":
        return store.load_facts(
            company_ticker, lookup.quarter, lookup.fy_start_year, "standalone"
        )
    return {}


import importlib

import capital_nerve_db
import capital_nerve_logic

importlib.reload(capital_nerve_db)
importlib.reload(capital_nerve_logic)
from capital_nerve_db import FactStore
from capital_nerve_logic import (
    accept_for_preferred_basis,
    dedupe_eps_values,
    earnings_card_metrics,
    interpret_metric_signals,
    validation_checks,
)

DB_PATH = DATA_DIR / "capital_nerve.db"
fact_store = FactStore(DB_PATH)


In [6]:
# ┌────────────────────────────┐
# │ Source Discovery Layer      │
# │ NSE/BSE/IR sites/manual     │
# └──────────────┬─────────────┘
#                │ filing.detected

_uploaded = globals().get("uploaded_candidates")
candidates = _uploaded if _uploaded else discover_filings()
filing_detected = emit(
    "filing.detected",
    {"filings": [asdict(c) for c in candidates]},
)

{
  "event_type": "filing.detected",
  "event_id": "evt_0ff7cff1ed96",
  "occurred_at": "2026-06-14T13:23:58.371939+00:00"
}

── filing.detected — cell output ──
{
  "filings": [
    {
      "source": "manual",
      "company_ticker": "Reliance Industries Ltd.",
      "title": "Q1-FY2024-25.pdf",
      "url": null,
      "local_path": "/Users/prairit/capital-nerve/v2/data/uploads/Q1-FY2024-25.pdf",
      "detected_at": "2026-06-14T13:23:58.369695+00:00"
    },
    {
      "source": "manual",
      "company_ticker": "Reliance Industries Ltd.",
      "title": "Q2-FY2024-25.pdf",
      "url": null,
      "local_path": "/Users/prairit/capital-nerve/v2/data/uploads/Q2-FY2024-25.pdf",
      "detected_at": "2026-06-14T13:23:58.369745+00:00"
    },
    {
      "source": "manual",
      "company_ticker": "Reliance Industries Ltd.",
      "title": "Q4_FY2025-26.pdf",
      "url": null,
      "local_path": "/Users/prairit/capital-nerve/v2/data/uploads/Q4_FY2025-26.pdf",
      "detected_at": "2026

In [7]:
# ┌────────────────────────────┐
# │ Document Ingestion Service  │
# │ download, hash, store       │
# └──────────────┬─────────────┘
#                │ document.stored

import shutil


@dataclass
class StoredDocument:
    document_id: str
    company_ticker: str
    source: str
    title: str
    content_type: str
    sha256: str
    storage_path: str
    byte_size: int
    ingested_at: str = field(default_factory=now_iso)


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def ingest_filing(filing: dict[str, Any]) -> StoredDocument:
    raw_dir = DATA_DIR / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    if filing.get("local_path"):
        src = Path(filing["local_path"])
        data = src.read_bytes()
        ext = src.suffix or ".bin"
    else:
        # Production: HTTP download from filing["url"]
        placeholder = f"Placeholder bytes for {filing['title']}".encode()
        data, ext = placeholder, ".pdf"

    digest = sha256_bytes(data)
    dest = raw_dir / f"{digest}{ext}"
    if not dest.exists():
        if filing.get("local_path"):
            shutil.copy2(filing["local_path"], dest)
        else:
            dest.write_bytes(data)

    return StoredDocument(
        document_id=new_id("doc"),
        company_ticker=filing["company_ticker"],
        source=filing["source"],
        title=filing["title"],
        content_type="application/pdf" if ext == ".pdf" else "text/markdown",
        sha256=digest,
        storage_path=str(dest),
        byte_size=len(data),
    )


stored_docs = [ingest_filing(f) for f in filing_detected["payload"]["filings"]]
document_stored = emit(
    "document.stored",
    {"documents": [asdict(d) for d in stored_docs]},
)

{
  "event_type": "document.stored",
  "event_id": "evt_0976cd02e7b4",
  "occurred_at": "2026-06-14T13:23:58.443899+00:00"
}

── document.stored — cell output ──
{
  "documents": [
    {
      "document_id": "doc_cf7b731d4b7c",
      "company_ticker": "Reliance Industries Ltd.",
      "source": "manual",
      "title": "Q1-FY2024-25.pdf",
      "content_type": "application/pdf",
      "sha256": "e29390caae95cc8f28606d9f08317cda424bf544fd86383c7f9ac7d25ca8e808",
      "storage_path": "/Users/prairit/capital-nerve/v2/data/raw/e29390caae95cc8f28606d9f08317cda424bf544fd86383c7f9ac7d25ca8e808.pdf",
      "byte_size": 1253337,
      "ingested_at": "2026-06-14T13:23:58.393067+00:00"
    },
    {
      "document_id": "doc_95c451f0bf93",
      "company_ticker": "Reliance Industries Ltd.",
      "source": "manual",
      "title": "Q2-FY2024-25.pdf",
      "content_type": "application/pdf",
      "sha256": "f78f4ade6ab7640fb74560b76505754fe5751c3602d61925c764c177875d1097",
      "storage_path": "

In [8]:
# ┌────────────────────────────┐
# │ Parsing / Markdown Service  │
# │ PDF → markdown/chunks       │
# └──────────────┬─────────────┘
#                │ document.parsed
# Requires: pip install openai python-dotenv pymupdf
# Config: .env (OPENAI_API_KEY; optional OPENAI_PDF_MODEL, OPENAI_PARSE_MAX_OUTPUT_TOKENS)
# PDFs are converted page-by-page (see capital_nerve_parse.py). Set FORCE_REPARSE=1 to rebuild cached .md

import importlib
import os
import re
from openai import OpenAI

import capital_nerve_parse

importlib.reload(capital_nerve_parse)

_TEXT_SUFFIXES = {".md", ".markdown", ".txt"}
OPENAI_PDF_MODEL = os.environ.get("OPENAI_PDF_MODEL", "gpt-4o")
PARSE_MAX_OUTPUT_TOKENS = int(os.environ.get("OPENAI_PARSE_MAX_OUTPUT_TOKENS", "8192"))
FORCE_REPARSE = os.environ.get("FORCE_REPARSE", "").lower() in ("1", "true", "yes")


@dataclass
class DocumentChunk:
    chunk_id: str
    document_id: str
    sequence: int
    heading: str | None
    text: str


@dataclass
class ParsedDocument:
    document_id: str
    markdown_path: str
    chunk_count: int
    chunks: list[DocumentChunk]
    reporting_period: ReportingPeriod | None = None
    preferred_metric_basis: str = PREFERRED_METRIC_BASIS
    available_bases: list[str] = field(default_factory=list)
    parsed_at: str = field(default_factory=now_iso)


def openai_client() -> OpenAI:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Add it to .env in the project root and re-run the bootstrap cell."
        )
    return OpenAI()


def is_pdf_file(path: Path, content_type: str) -> bool:
    if path.suffix.lower() == ".pdf" or content_type == "application/pdf":
        return True
    with path.open("rb") as fh:
        return fh.read(5) == b"%PDF-"


def pdf_to_markdown(storage_path: str) -> str:
    """Convert PDF to markdown — one LLM call per page."""
    path = Path(storage_path)
    total = capital_nerve_parse.pdf_page_count(path)
    print(f"  Parsing {path.name}: {total} page(s), model={OPENAI_PDF_MODEL}")

    def _on_page(page_num: int, page_total: int) -> None:
        print(f"    page {page_num}/{page_total}")

    return capital_nerve_parse.pdf_to_markdown_page_by_page(
        path,
        client=openai_client(),
        model=OPENAI_PDF_MODEL,
        max_output_tokens=PARSE_MAX_OUTPUT_TOKENS,
        on_page_done=_on_page,
    )


def to_markdown(storage_path: str, content_type: str) -> str:
    path = Path(storage_path)
    if is_pdf_file(path, content_type):
        return pdf_to_markdown(storage_path)
    if path.suffix.lower() in _TEXT_SUFFIXES:
        return path.read_text(encoding="utf-8", errors="replace")
    raise ValueError(f"Unsupported document type for {path.name}: {content_type}")


def chunk_markdown(document_id: str, markdown: str, max_chars: int = 1200) -> list[DocumentChunk]:
    sections = re.split(r"\n(?=#+\s)", markdown.strip()) or [markdown]
    chunks: list[DocumentChunk] = []
    for i, section in enumerate(sections):
        heading = section.splitlines()[0].lstrip("#").strip() if section.startswith("#") else None
        body = section if heading is None else "\n".join(section.splitlines()[1:]).strip()
        text = body or section
        for j in range(0, max(len(text), 1), max_chars):
            piece = text[j : j + max_chars]
            if not piece.strip():
                continue
            chunks.append(
                DocumentChunk(
                    chunk_id=new_id("chk"),
                    document_id=document_id,
                    sequence=len(chunks),
                    heading=heading,
                    text=piece.strip(),
                )
            )
    return chunks or [
        DocumentChunk(chunk_id=new_id("chk"), document_id=document_id, sequence=0, heading=None, text=markdown[:max_chars])
    ]


title_by_doc_id = {
    d["document_id"]: d.get("title", "")
    for d in document_stored["payload"]["documents"]
}

parsed_docs: list[ParsedDocument] = []
for doc in document_stored["payload"]["documents"]:
    storage_path = doc["storage_path"]
    parsed_dir = DATA_DIR / "parsed"
    parsed_dir.mkdir(parents=True, exist_ok=True)

    md_path = parsed_dir / f"{doc['document_id']}.md"
    meta_path = md_path.with_suffix(".meta.json")
    pdf_path = Path(storage_path)
    source_sha = doc.get("sha256")

    if capital_nerve_parse.should_reparse(
        md_path, pdf_path, source_sha256=source_sha, force=FORCE_REPARSE
    ):
        md = to_markdown(storage_path, doc["content_type"])
        issues = capital_nerve_parse.validate_parsed_markdown(md, pdf_path)
        if issues:
            print(f"  Parse validation warnings for {doc.get('title', doc['document_id'])}: {issues}")
        md_path.write_text(md, encoding="utf-8")
        if is_pdf_file(pdf_path, doc["content_type"]):
            capital_nerve_parse.write_parse_meta(
                meta_path,
                source_sha256=source_sha or "",
                page_count=capital_nerve_parse.pdf_page_count(pdf_path),
            )
    else:
        md = md_path.read_text(encoding="utf-8")
        print(f"  Using cached markdown: {md_path.name}")

    chunks = chunk_markdown(doc["document_id"], md)
    reporting_period = detect_reporting_period(
        md, title_by_doc_id.get(doc["document_id"], "")
    )
    parsed_docs.append(
        ParsedDocument(
            document_id=doc["document_id"],
            markdown_path=str(md_path),
            chunk_count=len(chunks),
            chunks=chunks,
            reporting_period=reporting_period,
        )
    )

_parsed_payload = {
    "documents": [
        {
            "document_id": p.document_id,
            "markdown_path": p.markdown_path,
            "chunk_count": p.chunk_count,
            "parsed_at": p.parsed_at,
            "reporting_period": (
                p.reporting_period.to_dict() if p.reporting_period else None
            ),
            "preferred_metric_basis": p.preferred_metric_basis,
            "available_bases": p.available_bases,
            "chunks": [asdict(c) for c in p.chunks],
        }
        for p in parsed_docs
    ]
}
print_cell_output("parsed documents (pre-emit)", _parsed_payload)
document_parsed = emit("document.parsed", _parsed_payload)

  Parsing e29390caae95cc8f28606d9f08317cda424bf544fd86383c7f9ac7d25ca8e808.pdf: 37 page(s), model=gpt-4o
    page 1/37
    page 2/37
    page 3/37
    page 4/37
    page 5/37
    page 6/37
    page 7/37
    page 8/37
    page 9/37
    page 10/37
    page 11/37
    page 12/37
    page 13/37
    page 14/37
    page 15/37
    page 16/37
    page 17/37
    page 18/37
    page 19/37
    page 20/37
    page 21/37
    page 22/37
    page 23/37
    page 24/37
    page 25/37
    page 26/37
    page 27/37
    page 28/37
    page 29/37
    page 30/37
    page 31/37
    page 32/37
    page 33/37
    page 34/37
    page 35/37
    page 36/37
    page 37/37
  Parsing f78f4ade6ab7640fb74560b76505754fe5751c3602d61925c764c177875d1097.pdf: 44 page(s), model=gpt-4o
    page 1/44
    page 2/44
    page 3/44
    page 4/44
    page 5/44
    page 6/44
    page 7/44
    page 8/44
    page 9/44
    page 10/44
    page 11/44
    page 12/44
    page 13/44
    page 14/44
    page 15/44
    page 16/44
    page 17/4

In [9]:
# ┌────────────────────────────┐
# │ Extraction Service          │
# │ OpenAI LLM extraction       │
# └──────────────┬─────────────┘
#                │ values.extracted
# Config: .env (OPENAI_API_KEY, optional OPENAI_EXTRACT_MODEL or OPENAI_PDF_MODEL)

import importlib

import capital_nerve_logic

importlib.reload(capital_nerve_logic)
from capital_nerve_logic import (
    CANONICAL_UNIT_ENUM,
    dedupe_eps_values,
    find_chunk_id,
    resolve_unit,
)

import json
import os
import re as _re
from openai import OpenAI

OPENAI_EXTRACT_MODEL = (
    os.environ.get("OPENAI_EXTRACT_MODEL")
    or os.environ.get("OPENAI_PDF_MODEL")
    or "gpt-4o"
)

from catalog_loader import allowed_extraction_keys, canonical_fact_key

ALLOWED_FACT_KEYS = sorted(set(allowed_extraction_keys()) | {"interest_earned"})

EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "values": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "fact_key": {"type": "string", "enum": ALLOWED_FACT_KEYS},
                    "raw_value": {"type": "string"},
                    "unit": {"type": ["string", "null"]},
                    "period": {"type": ["string", "null"]},
                    "basis": {
                        "type": ["string", "null"],
                        "enum": ["standalone", "consolidated", None],
                    },
                    "evidence": {"type": "string"},
                    "confidence": {"type": "number"},
                },
                "required": [
                    "fact_key",
                    "raw_value",
                    "unit",
                    "period",
                    "basis",
                    "evidence",
                    "confidence",
                ],
                "additionalProperties": False,
            },
        }
    },
    "required": ["values"],
    "additionalProperties": False,
}


@dataclass
class ExtractedValue:
    value_id: str
    document_id: str
    chunk_id: str
    fact_key: str
    raw_value: str
    numeric_value: float | None
    unit: str | None
    chunk_unit_hint: str | None
    period: str | None
    basis: str | None
    evidence: str
    extractor: str  # slm | llm | ocr | table_parser | regex
    confidence: float


def extraction_client() -> OpenAI:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Add it to .env and re-run the bootstrap cell."
        )
    return OpenAI()


def parse_number(raw: str | None) -> float | None:
    if not raw:
        return None
    try:
        return float(str(raw).replace(",", "").strip())
    except ValueError:
        return None


def parse_llm_payload(content: str) -> list[dict[str, Any]]:
    text = content.strip()
    if text.startswith("```"):
        text = _re.sub(r"^```(?:json)?\s*", "", text)
        text = _re.sub(r"\s*```$", "", text)
    data = json.loads(text)
    if isinstance(data, dict):
        values = data.get("values", [])
        return values if isinstance(values, list) else []
    return data if isinstance(data, list) else []


def resolve_basis(
    entry: dict[str, Any],
    chunks: list[dict[str, Any]],
    evidence: str,
    raw_value: str,
) -> str | None:
    basis = entry.get("basis")
    if basis:
        return str(basis).strip().lower()
    chunk_id = find_chunk_id(chunks, evidence, str(raw_value))
    for chunk in chunks:
        if chunk["chunk_id"] == chunk_id:
            return basis_from_heading(chunk.get("heading"))
    return None


def extract_from_document(doc: dict[str, Any], client: OpenAI) -> list[ExtractedValue]:
    markdown = Path(doc["markdown_path"]).read_text(encoding="utf-8")
    chunks = doc["chunks"]
    rp = doc.get("reporting_period")
    preferred_basis = doc.get("preferred_metric_basis", PREFERRED_METRIC_BASIS)
    period_hint = ""
    if rp:
        period_hint = (
            f"\nPrimary reporting period for this filing: {rp['label']} "
            f"(quarter ended {rp['quarter_end']}). Tag current-quarter column values with this label.\n"
        )
    basis_hint = (
        f"\nPreferred metric basis: {preferred_basis} (investor/group view). "
        f"Tag each value with basis \"standalone\" or \"consolidated\" from its section heading.\n"
    )
    if preferred_basis == "consolidated":
        basis_hint += (
            "Prioritize consolidated / group results tables; include standalone only if "
            "no consolidated figure exists for that metric and period.\n"
        )
    prompt = f"""Extract financial metrics from this Indian corporate filing markdown.
            {period_hint}{basis_hint}
            Return one entry per distinct metric value you find (include multiple periods/columns when present).
            Use these fact_key values only: {", ".join(ALLOWED_FACT_KEYS)}.

            Guidance:
            - revenue: total income, segment revenue, or explicit revenue line items
            - net_profit: net profit / loss for the period
            - eps: basic earnings per share only (one row per period; skip diluted unless basic missing)
            - ebitda: only if explicitly stated; do not infer
            - interest_earned / operating_profit: when explicitly labeled in the filing
            - unit: use only enum values — P&L/balance-sheet amounts take unit from table header ("in crores" → "crore"); EPS is always "Rs"; do not use INR_cr or pct
            - period: e.g. "Q3 FY26", "31.12.2025", "nine months ended 31.12.2025"
            - basis: "standalone" or "consolidated" — required; infer from section title if needed
            - evidence: short verbatim snippet containing the number
            - confidence: 0.0-1.0 based on label clarity

            Filing markdown:
            ---
            {markdown[:120000]}
            ---"""

    response = client.chat.completions.create(
        model=OPENAI_EXTRACT_MODEL,
        messages=[
            {
                "role": "developer",
                "content": "You extract structured financial metrics from filings. Return JSON only.",
            },
            {"role": "user", "content": prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "extracted_values",
                "strict": True,
                "schema": EXTRACTION_SCHEMA,
            },
        },
        temperature=0,
    )

    content = response.choices[0].message.content or "{}"
    entries = parse_llm_payload(content)

    found: list[ExtractedValue] = []
    for entry in entries:
        raw_key = entry.get("fact_key") or entry.get("metric_key")
        fact_key = canonical_fact_key(raw_key) or raw_key
        raw_value = entry.get("raw_value")
        if not fact_key or not raw_value:
            continue
        evidence = (entry.get("evidence") or "").strip()
        resolved_unit, unit_hint = resolve_unit(
            entry, chunks, evidence, str(raw_value), fact_key
        )
        found.append(
            ExtractedValue(
                value_id=new_id("val"),
                document_id=doc["document_id"],
                chunk_id=find_chunk_id(chunks, evidence, str(raw_value)),
                fact_key=fact_key,
                raw_value=str(raw_value),
                numeric_value=parse_number(str(raw_value)),
                unit=resolved_unit,
                chunk_unit_hint=unit_hint,
                period=entry.get("period"),
                basis=resolve_basis(entry, chunks, evidence, str(raw_value)),
                evidence=evidence or str(raw_value),
                extractor="llm",
                confidence=float(entry.get("confidence") or 0.85),
            )
        )
    return dedupe_eps_values(found)


client = extraction_client()
extracted_values: list[ExtractedValue] = []
for doc in document_parsed["payload"]["documents"]:
    extracted_values.extend(extract_from_document(doc, client))

values_extracted = emit(
    "values.extracted",
    {"values": [asdict(v) for v in extracted_values], "count": len(extracted_values)},
)

{
  "event_type": "values.extracted",
  "event_id": "evt_d65db2373e69",
  "occurred_at": "2026-06-14T14:18:36.190650+00:00"
}

── values.extracted — cell output ──
{
  "values": [
    {
      "value_id": "val_eae88a3bf1b0",
      "document_id": "doc_cf7b731d4b7c",
      "chunk_id": "chk_5b0267176b1a",
      "fact_key": "revenue_from_operations",
      "raw_value": "257,823",
      "numeric_value": 257823.0,
      "unit": "crore",
      "chunk_unit_hint": "crore",
      "period": "Q1 FY24",
      "basis": "consolidated",
      "evidence": "QUARTERLY CONSOLIDATED REVENUE AT \u20b9 257,823 CRORE",
      "extractor": "llm",
      "confidence": 1.0
    },
    {
      "value_id": "val_cdd9846c13da",
      "document_id": "doc_cf7b731d4b7c",
      "chunk_id": "chk_5b0267176b1a",
      "fact_key": "ebitda",
      "raw_value": "42,748",
      "numeric_value": 42748.0,
      "unit": "crore",
      "chunk_unit_hint": "crore",
      "period": "Q1 FY24",
      "basis": "consolidated",
      "evidenc

In [10]:
# ┌────────────────────────────┐
# │ Validation / Trust Service  │
# │ evidence, formula, checks   │
# └──────────────┬─────────────┘
#                │ values.validated

import importlib

import capital_nerve_logic

importlib.reload(capital_nerve_logic)
from catalog_loader import canonical_fact_key
from capital_nerve_logic import (
    accept_for_preferred_basis,
    canonicalize_unit,
    is_blocking_check,
    validation_checks,
)


@dataclass
class ValidatedValue:
    value_id: str
    fact_key: str
    numeric_value: float
    unit: str | None
    period: str | None
    basis: str | None
    status: str  # accepted | rejected | needs_review
    checks: list[str]
    evidence: str


def validate_value(
    row: dict[str, Any], preferred_basis: str = PREFERRED_METRIC_BASIS
) -> ValidatedValue | None:
    checks = validation_checks(row, preferred_basis)
    if "non_numeric" in checks:
        return ValidatedValue(
            value_id=row["value_id"],
            fact_key=row["fact_key"],
            numeric_value=0.0,
            unit=row.get("unit"),
            period=row.get("period"),
            basis=row.get("basis"),
            status="rejected",
            checks=checks,
            evidence=row["evidence"],
        )

    v = float(row["numeric_value"])
    blocking = [c for c in checks if is_blocking_check(c)]
    status = "accepted" if not blocking else "needs_review"
    unit = canonicalize_unit(row.get("unit"))

    return ValidatedValue(
        value_id=row["value_id"],
        fact_key=row["fact_key"],
        numeric_value=v,
        unit=unit,
        period=row.get("period"),
        basis=row.get("basis"),
        status=status,
        checks=checks,
        evidence=row["evidence"],
    )


def _primary_from_parsed(field: str) -> Any:
    for doc in document_parsed["payload"]["documents"]:
        if field in doc and doc[field] is not None:
            return doc[field]
    return None


def _latest_reporting_period() -> dict[str, Any] | None:
    periods = [
        doc["reporting_period"]
        for doc in document_parsed["payload"]["documents"]
        if doc.get("reporting_period")
    ]
    if not periods:
        return None
    return max(periods, key=lambda p: p["quarter_end"])


def _company_ticker() -> str:
    for doc in document_stored["payload"]["documents"]:
        return doc["company_ticker"]
    for filing in filing_detected["payload"]["filings"]:
        return filing["company_ticker"]
    return "UNKNOWN"


def _document_by_id(document_id: str) -> dict[str, Any] | None:
    for doc in document_stored["payload"]["documents"]:
        if doc["document_id"] == document_id:
            return doc
    return None


def persist_validated_values(
    accepted_rows: list[dict[str, Any]],
    *,
    company_ticker: str,
    primary_period: dict[str, Any] | None,
    preferred_basis: str,
) -> dict[str, Any]:
    persisted = 0
    for doc in document_stored["payload"]["documents"]:
        rp = None
        for parsed in document_parsed["payload"]["documents"]:
            if parsed["document_id"] == doc["document_id"]:
                rp = parsed.get("reporting_period")
                break
        period = ReportingPeriod.from_dict(rp) if rp else None
        fact_store.upsert_filing(
            document_id=doc["document_id"],
            company_ticker=doc["company_ticker"],
            sha256=doc["sha256"],
            title=doc["title"],
            quarter=period.quarter if period else None,
            fy_start_year=period.fy_start_year if period else None,
            fy_label=period.fy_label if period else None,
            quarter_end=period.quarter_end if period else None,
            ingested_at=doc["ingested_at"],
        )

    seen: set[tuple[int, int, str, str]] = set()
    for row in accepted_rows:
        rp = resolve_period_label(row.get("period"))
        if not rp and primary_period:
            if period_match_score(row.get("period"), ReportingPeriod.from_dict(primary_period)) > 0:
                rp = ReportingPeriod.from_dict(primary_period)
        if not rp:
            continue
        basis = (row.get("basis") or preferred_basis).strip().lower()
        key = (rp.quarter, rp.fy_start_year, row["fact_key"], basis)
        if key in seen:
            continue
        seen.add(key)
        source_doc = _document_by_id(row.get("document_id", ""))
        fact_store.upsert_fact(
            company_ticker=company_ticker,
            quarter=rp.quarter,
            fy_start_year=rp.fy_start_year,
            fy_label=rp.fy_label,
            quarter_end=rp.quarter_end,
            fact_key=canonical_fact_key(row["fact_key"]) or row["fact_key"],
            basis=basis,
            numeric_value=float(row["numeric_value"]),
            unit=row.get("unit"),
            evidence=row.get("evidence"),
            source_document_id=source_doc["document_id"] if source_doc else None,
            status=row.get("status", "accepted"),
        )
        persisted += 1

    return {
        "company_ticker": company_ticker,
        "persisted_count": persisted,
        "periods_in_db": fact_store.list_periods(company_ticker, preferred_basis),
    }


_reporting_period = _latest_reporting_period() or _primary_from_parsed("reporting_period")
_preferred_basis = (
    _primary_from_parsed("preferred_metric_basis") or PREFERRED_METRIC_BASIS
)
_company = _company_ticker()

validated_values = [
    validate_value(v, _preferred_basis)
    for v in values_extracted["payload"]["values"]
]
validated_values = [v for v in validated_values if v is not None]
accepted = accept_for_preferred_basis(validated_values, _preferred_basis)

# Re-attach document_id from extraction for provenance
_extracted_by_id = {v["value_id"]: v for v in values_extracted["payload"]["values"]}
_accepted_rows = []
for v in accepted:
    row = asdict(v)
    row["document_id"] = _extracted_by_id.get(v.value_id, {}).get("document_id")
    _accepted_rows.append(row)

_persist_result = persist_validated_values(
    _accepted_rows,
    company_ticker=_company,
    primary_period=_reporting_period,
    preferred_basis=_preferred_basis,
)

_validated_payload = {
    "reporting_period": _reporting_period,
    "preferred_metric_basis": _preferred_basis,
    "available_bases": _primary_from_parsed("available_bases") or [],
    "values": [asdict(v) for v in validated_values],
    "accepted_count": len(accepted),
    "persisted": _persist_result,
}
print_cell_output("validated values (pre-emit)", _validated_payload)
values_validated = emit("values.validated", _validated_payload)



── validated values (pre-emit) ──
{
  "reporting_period": {
    "quarter": 4,
    "fiscal_year": 26,
    "quarter_end": "2026-03-31",
    "label": "Q4 FY26",
    "period_type": "quarter",
    "source": "heading"
  },
  "preferred_metric_basis": "consolidated",
  "available_bases": [],
  "values": [
    {
      "value_id": "val_eae88a3bf1b0",
      "fact_key": "revenue_from_operations",
      "numeric_value": 257823.0,
      "unit": "crore",
      "period": "Q1 FY24",
      "basis": "consolidated",
      "status": "accepted",
      "checks": [],
      "evidence": "QUARTERLY CONSOLIDATED REVENUE AT \u20b9 257,823 CRORE"
    },
    {
      "value_id": "val_cdd9846c13da",
      "fact_key": "ebitda",
      "numeric_value": 42748.0,
      "unit": "crore",
      "period": "Q1 FY24",
      "basis": "consolidated",
      "status": "accepted",
      "checks": [],
      "evidence": "QUARTERLY CONSOLIDATED EBITDA AT \u20b9 42,748 CRORE"
    },
    {
      "value_id": "val_a8722af620b0",
      "fa

In [11]:
# ┌────────────────────────────┐
# │ Metrics Engine              │
# │ catalog/metrics.json        │
# └──────────────┬─────────────┘
#                │ metrics.calculated

import importlib

import capital_nerve_logic
import catalog_engine

importlib.reload(catalog_engine)
importlib.reload(capital_nerve_logic)
from capital_nerve_logic import build_raw_details, compute_pipeline_metrics

_rp_raw = values_validated["payload"].get("reporting_period")
_preferred_basis = values_validated["payload"].get(
    "preferred_metric_basis", PREFERRED_METRIC_BASIS
)
_company = values_validated["payload"].get("persisted", {}).get("company_ticker", "UNKNOWN")
reporting_period = ReportingPeriod.from_dict(_rp_raw) if _rp_raw else None
_all_values = values_validated["payload"]["values"]

if reporting_period:
    base = resolve_metrics(
        _all_values,
        reporting_period,
        _preferred_basis,
        fact_store,
        _company,
    )
    prior_yoy = resolve_metrics(
        _all_values,
        reporting_period,
        _preferred_basis,
        fact_store,
        _company,
        prior_year=True,
    )
    prior_qoq = resolve_metrics(
        _all_values,
        reporting_period,
        _preferred_basis,
        fact_store,
        _company,
        prior_quarter=True,
    )
else:
    base = {}
    prior_yoy = {}
    prior_qoq = {}
    for row in _all_values:
        if row.get("status") != "accepted":
            continue
        base[row["fact_key"]] = float(row["numeric_value"])


def _resolve_prov(fact_key: str, scope: str) -> dict[str, Any] | None:
    if not reporting_period:
        return None
    return resolve_input_provenance(
        _all_values,
        reporting_period,
        _preferred_basis,
        fact_store,
        _company,
        fact_key,
        prior_year=scope.upper() == "PY",
        prior_quarter=scope.upper() == "PQ",
    )


raw_details = build_raw_details(_all_values)
metrics = compute_pipeline_metrics(
    base,
    prior_yoy,
    prior_qoq,
    period_label=reporting_period.label if reporting_period else None,
    raw_details=raw_details,
    resolve_provenance=_resolve_prov if reporting_period else None,
)
for m in metrics:
    m.setdefault("metric_id", new_id("met"))

_metrics_basis_used = _preferred_basis
if reporting_period:
    _tgt = (
        reporting_period
        if isinstance(reporting_period, ReportingPeriod)
        else ReportingPeriod.from_dict(reporting_period)
    )
    if not _select_metrics(_all_values, _tgt, _preferred_basis, allow_fallback=False):
        if (
            _preferred_basis == "consolidated"
            and _select_metrics(_all_values, _tgt, "standalone", allow_fallback=False)
        ):
            _metrics_basis_used = "standalone"

metrics_calculated = emit(
    "metrics.calculated",
    {
        "reporting_period": _rp_raw,
        "preferred_metric_basis": _preferred_basis,
        "metrics_basis_used": _metrics_basis_used,
        "company_ticker": _company,
        "metrics": metrics,
    },
)


NameError: name 'metric_key' is not defined

In [ ]:
# ┌────────────────────────────┐
# │ Signal Engine               │
# │ interpretation layer        │
# └──────────────┬─────────────┘
#                │ signals.generated

import importlib

import capital_nerve_logic
from catalog_loader import get_catalog

importlib.reload(capital_nerve_logic)
from capital_nerve_logic import interpret_metric_signals


@dataclass
class Signal:
    signal_id: str
    signal_key: str
    severity: str  # info | watch | alert
    headline: str
    rationale: str
    metric_keys: list[str]
    category: str | None = None
    direction: str | None = None


def interpret_metrics(metrics: list[dict[str, Any]]) -> list[Signal]:
    return [
        Signal(
            signal_id=new_id("sig"),
            signal_key=payload["signal_key"],
            severity=payload["severity"],
            headline=payload["headline"],
            rationale=payload["rationale"],
            metric_keys=payload.get("metric_keys") or [],
            category=payload.get("category"),
            direction=payload.get("direction"),
        )
        for payload in interpret_metric_signals(metrics)
    ]


_mc = metrics_calculated["payload"]
signals = interpret_metrics(_mc["metrics"])
_signal_payloads = [asdict(s) for s in signals]

_rp = _mc.get("reporting_period") or {}
_persist_result = None
if _rp.get("quarter") and (_rp.get("fy_start_year") or _rp.get("fiscal_year")) and _rp.get("quarter_end"):
    _rp_norm = ReportingPeriod.from_dict(_rp)
    _persist_result = fact_store.persist_period_signals(
        company_ticker=_mc.get("company_ticker", "UNKNOWN"),
        quarter=_rp_norm.quarter,
        fy_start_year=_rp_norm.fy_start_year,
        fy_label=_rp_norm.fy_label,
        quarter_end=str(_rp["quarter_end"]),
        basis=_mc.get("metrics_basis_used") or _mc.get("preferred_metric_basis") or PREFERRED_METRIC_BASIS,
        signals=_signal_payloads,
        metrics=_mc["metrics"],
        catalog_version=get_catalog().version,
    )

signals_generated = emit(
    "signals.generated",
    {
        "signals": _signal_payloads,
        "persisted": _persist_result,
    },
)


In [ ]:
# ┌────────────────────────────┐
# │ Card Engine                 │
# │ structured intelligence     │
# └──────────────┬─────────────┘
#                │ cards.generated

import importlib

import capital_nerve_logic

importlib.reload(capital_nerve_logic)
from capital_nerve_logic import earnings_card_metrics


@dataclass
class IntelligenceCard:
    card_id: str
    card_type: str
    title: str
    summary: str
    metrics: list[dict[str, Any]]
    signals: list[dict[str, Any]]
    reporting_period: dict[str, Any] | None = None
    preferred_metric_basis: str | None = None
    generated_at: str = field(default_factory=now_iso)


def build_cards(
    metrics: list[dict[str, Any]],
    sigs: list[dict[str, Any]],
    reporting_period: dict[str, Any] | None,
    preferred_metric_basis: str | None,
    *,
    metrics_basis_used: str | None = None,
    company_ticker: str | None = None,
    store: FactStore | None = None,
) -> list[IntelligenceCard]:
    top_metrics = earnings_card_metrics(metrics)
    period_label = reporting_period["label"] if reporting_period else None
    basis_label = metrics_basis_used or preferred_metric_basis or PREFERRED_METRIC_BASIS
    if period_label:
        period_title = f"Earnings snapshot — {period_label} ({basis_label})"
    else:
        period_title = "Earnings intelligence snapshot"
    cards: list[IntelligenceCard] = [
        IntelligenceCard(
            card_id=new_id("card"),
            card_type="earnings_snapshot",
            title=period_title,
            summary="; ".join(s["headline"] for s in sigs) or "Pipeline run complete",
            metrics=top_metrics,
            signals=sigs,
            reporting_period=reporting_period,
            preferred_metric_basis=preferred_metric_basis,
        )
    ]

    if store and company_ticker:
        revenue_trend = store.get_trend_alias_aware(company_ticker, "revenue_from_operations", basis_label, n=4)
        if len(revenue_trend) >= 2:
            cards.append(
                IntelligenceCard(
                    card_id=new_id("card"),
                    card_type="metric_trend",
                    title=f"Revenue trend — last {len(revenue_trend)} quarters",
                    summary=f"Revenue ({basis_label}) across stored quarters",
                    metrics=[
                        {
                            "metric_key": "revenue_from_operations",
                            "value": pt["value"],
                            "unit": "INR_cr",
                            "derivation": "raw",
                            "period": pt["label"],
                            "evidence": pt.get("evidence"),
                            "source": pt.get("source", "database"),
                        }
                        for pt in revenue_trend
                    ],
                    signals=[],
                    reporting_period=reporting_period,
                    preferred_metric_basis=preferred_metric_basis,
                )
            )
    return cards


cards = build_cards(
    metrics_calculated["payload"]["metrics"],
    signals_generated["payload"]["signals"],
    metrics_calculated["payload"].get("reporting_period"),
    metrics_calculated["payload"].get("preferred_metric_basis"),
    metrics_basis_used=metrics_calculated["payload"].get("metrics_basis_used"),
    company_ticker=metrics_calculated["payload"].get("company_ticker"),
    store=fact_store,
)
cards_generated = emit(
    "cards.generated",
    {
        "reporting_period": metrics_calculated["payload"].get("reporting_period"),
        "preferred_metric_basis": metrics_calculated["payload"].get(
            "preferred_metric_basis"
        ),
        "metrics_basis_used": metrics_calculated["payload"].get(
            "metrics_basis_used"
        ),
        "company_ticker": metrics_calculated["payload"].get("company_ticker"),
        "cards": [asdict(c) for c in cards],
    },
)

In [ ]:
# ┌────────────────────────────┐
# │ Serving Layer              │
# │ UI / API / alerts / search │
# └────────────────────────────┘


def serve_snapshot(
    cards_event: dict[str, Any],
    store: FactStore | None = None,
) -> dict[str, Any]:
    """Materialize API/UI/search payloads from generated cards."""
    company = cards_event["payload"].get("company_ticker")
    basis = (
        cards_event["payload"].get("metrics_basis_used")
        or cards_event["payload"].get("preferred_metric_basis")
        or PREFERRED_METRIC_BASIS
    )
    trends: dict[str, Any] = {}
    history: list[dict[str, Any]] = []
    if store and company:
        for metric_key in ("revenue_from_operations", "pat", "operating_profit"):
            series = store.get_trend_alias_aware(company, metric_key, basis, n=8)
            if series:
                trends[metric_key] = series
        history = store.list_periods(company, basis)

    snapshot = {
        "served_at": now_iso(),
        "reporting_period": cards_event["payload"].get("reporting_period"),
        "preferred_metric_basis": cards_event["payload"].get(
            "preferred_metric_basis"
        )
        or PREFERRED_METRIC_BASIS,
        "metrics_basis_used": basis,
        "company_ticker": company,
        "cards": cards_event["payload"]["cards"],
        "history": history,
        "trends": trends,
        "endpoints": {
            "rest": "/api/v1/cards",
            "search": "/api/v1/search",
            "alerts": "/api/v1/alerts",
            "trends": "/api/v1/metrics/trend",
        },
    }
    out_path = DATA_DIR / "serving_snapshot.json"
    out_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    return snapshot


serving_snapshot = serve_snapshot(cards_generated, fact_store)
print("Serving snapshot written to", DATA_DIR / "serving_snapshot.json")
print("Fact store:", DB_PATH.resolve())
print_cell_output("serving — snapshot", serving_snapshot)